In [1]:
#library for retreiving data from dotenv file
import os
from dotenv import load_dotenv

#for model building using OpenAi
from openai import OpenAI

load_dotenv()
api_key=os.environ.get("GROQ_API_KEY")

In [2]:
from langchain_core.prompts import PromptTemplate

prompt_temp_name = PromptTemplate(
    input_variables = ['cuisine'],
    template = "I want to open a restaurant for {cuisine} food , suggest me 10 cool names. give just names not any explanation"
)

prompt_temp_name.format(cuisine = "mexican")

'I want to open a restaurant for mexican food , suggest me 10 cool names. give just names not any explanation'

## Chaining

In [3]:
from langchain_core.prompts import ChatPromptTemplate #for prompts 
from langchain_core.output_parsers import StrOutputParser # for structured o/p
from langchain_groq import ChatGroq #llm

#initialize basic llm using llama
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7
)

#create prompt template
chat_prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple terms and bullet points , not too long just ion 5 lines."
)


parser = StrOutputParser()

#this is chain using pipe operator : prompt-> llm -> o/p
chain = chat_prompt | llm | parser

response = chain.invoke({"topic" : "Generative AI"})

print(response)

Generative AI creates new content, such as:
* Images
* Text
* Music
* Videos
It uses algorithms to generate original material, mimicking human creativity.


In [4]:
# MultiChaining -> one than one prompts layers

explain_prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in detail."
)

summary_prompt = ChatPromptTemplate.from_template(
    "Summarize this text in 3 bullet points:\n\n{text}"
)

explanation_chain = explain_prompt | llm | parser

summary_chain = summary_prompt | llm | parser

#this is modern sequential chain
full_chain = explanation_chain | summary_chain

result = full_chain.invoke({"topic": "Transformers in AI"})

print(result)

Here are three bullet points summarizing the text:

* Transformers are a type of neural network architecture that use self-attention mechanisms to process input sequences, such as text or images, and have become a staple in many state-of-the-art models for tasks like language translation, text classification, and question answering.
* The key components of a Transformer include the self-attention mechanism, encoder-decoder architecture, multi-head attention, and positional encoding, which allow the model to weigh the importance of different input elements, attend to different parts of the input sequence, and preserve the order of the input sequence.
* Transformers have several advantages, including parallelization, scalability, and flexibility, and have been widely used in various applications such as language translation, text classification, question answering, and image captioning, with real-world examples including Google Translate, BERT, and RoBERTa, but also have limitations such

In [5]:
# Sequential Chain with multiple o/p
from langchain_core.runnables import RunnableParallel # it helps for multiple outputs at same time in form of dictionary

# chain = {
#     "restaurants": restaurant_chain,
#     "food_items": food_chain
# }

restaurant_prompt = ChatPromptTemplate.from_template(
    "Suggest 5 popular restaurants for {cuisine} cuisine. Give just name not any explanation."
)
restaurant_chain = restaurant_prompt | llm | parser

food_prompt = ChatPromptTemplate.from_template(
    "List 5 famous food items from {cuisine} cuisine. Give just name not any explanation."
)
food_chain = food_prompt | llm | parser

multi_output_chain = RunnableParallel(
    restaurants=restaurant_chain,
    food_items=food_chain
)
result = multi_output_chain.invoke({"cuisine": "Indian"})

print("Restaurants:\n", result["restaurants"])
print("\nFood Items:\n", result["food_items"])

Restaurants:
 1. Tandoori Nights
2. Taj Mahal
3. Dosa Place
4. Spice Affair
5. Karim's

Food Items:
 1. Tandoori Chicken
2. Biryani
3. Naan
4. Samosa
5. Gulab Jamun


## Tooling and Agents

In [6]:
#Creating Tools 

from langchain.tools import tool

@tool
def restaurant_finder(cuisine : str) -> str:
    """Returns popular restaurants for a given cuisine."""
    data = {
        "italian": ["Olive Garden", "La Pinoz Pizza", "Little Italy"],
        "indian": ["Bukhara", "Karim's", "Indian Accent"],
        "japanese": ["Sakura", "Megu", "Kofuku"]
    }
    return ", ".join(data.get(cuisine.lower(), ["No restaurants found"]))

@tool
def food_items(cuisine: str) -> str:
    """Returns popular food items from a cuisine."""

    data = {
        "italian": ["Pizza", "Pasta", "Risotto"],
        "indian": ["Butter Chicken", "Biryani", "Paneer Tikka"],
        "japanese": ["Sushi", "Ramen", "Tempura"]
    }

    return ", ".join(data.get(cuisine.lower(), ["No items found"]))


#registering tools
tools = [restaurant_finder, food_items]

In [7]:
#Creating an Agent

from langchain.agents import create_agent

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that recommends restaurants and food items."),
    ("human", "{input}")
])

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant that recommends restaurants and food items."
)


response = agent.invoke({
    "messages": [{"role": "user", "content": "Suggest italian restaurants and food items"}]
})
print(response["messages"][-1].content)

You can try Olive Garden, La Pinoz Pizza, or Little Italy for Italian restaurants. They serve a variety of dishes including pizza, pasta, and risotto. Would you like more recommendations or details about these options?


In [8]:
for msg in response["messages"]:
    print(type(msg).__name__, ":", msg.content)

HumanMessage : Suggest italian restaurants and food items
AIMessage : 
ToolMessage : Olive Garden, La Pinoz Pizza, Little Italy
ToolMessage : Pizza, Pasta, Risotto
AIMessage : You can try Olive Garden, La Pinoz Pizza, or Little Italy for Italian restaurants. They serve a variety of dishes including pizza, pasta, and risotto. Would you like more recommendations or details about these options?


In [9]:
for msg in response["messages"]:
    if hasattr(msg, "tool_calls"):
        print(msg.tool_calls)

[{'name': 'restaurant_finder', 'args': {'cuisine': 'italian'}, 'id': 'ca3z8r33b', 'type': 'tool_call'}, {'name': 'food_items', 'args': {'cuisine': 'italian'}, 'id': '1azc11a3g', 'type': 'tool_call'}]
[]


In [10]:
#result in other way
result = agent.invoke({
    "messages": [{"role": "user", "content": "Suggest indian restaurants"}]
})

final_answer = result["messages"][-1].content

print(final_answer)

Some popular Indian restaurants include Bukhara, Karim's, and Indian Accent. Would you like more suggestions or information about these restaurants?


## Memory

In [12]:
chat_prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple terms and bullet points , not too long just in 5 lines."
)

#this is chain using pipe operator : prompt-> llm -> o/p
chain = chat_prompt | llm | parser

print(chain.invoke({'topic' : 'Gen AI'}))

Gen AI refers to General Artificial Intelligence. 
Key points:
* Intelligent machines that learn
* Perform tasks like humans
* Can reason and solve problems
* Continuously improve themselves
* May surpass human intelligence.


In [16]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

response = llm.invoke([
    HumanMessage(content="My name is Anubhav")
])

chat_history.append(HumanMessage(content="My name is Anubhav"))
chat_history.append(AIMessage(content=response.content))

response2 = llm.invoke(
    chat_history + [HumanMessage(content="What is my name?")]
)
print(response2.content)

Your name is Anubhav.


In [18]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain  = prompt | llm | parser

store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)
conversation.invoke(
    {"input": "Hi my name is Anubhav"},
    config={"configurable": {"session_id": "1"}}
)

conversation.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": "1"}}
)

'Your name is Anubhav.'